In [ ]:
from ultralytics import YOLO
from matplotlib import image
from matplotlib import pyplot as plt
from PIL import Image

# from IPython import display
# display.clear_output()
# from IPython.display import display, Image
import glob
import random
import os

HOME = os.getcwd()

In [ ]:
# pre-trained model for detection
model_detection = YOLO("yolov8n.pt")
# pre-trained model for segmentation
model_segmentation = YOLO("yolov8n-seg.pt")

In [ ]:
Test_Image4 = "../images/cat.jpg"
# Image(Test_Image4)
# run the same models:
prediction_detection = model_detection.predict(Test_Image4)
prediction_segmentation = model_segmentation.predict(Test_Image4)
result_array = []
result_array.append(prediction_detection[0].plot())
result_array.append(prediction_segmentation[0].plot())
fig = plt.figure(figsize=(20, 10))
ax1 = fig.add_subplot(1, 2, 1)
ax1.set_title("Detection Result")
ax1.imshow(result_array[0])
ax2 = fig.add_subplot(1, 2, 2)
ax2.set_title("Segmentation Result")
ax2.imshow(result_array[1])

In [ ]:
from collections import Counter

# prediction_detection is a list of Results objects (one for each image in the batch)
# Since you're predicting on a single image, you'll access the first (and only) Results object
results = prediction_detection[0]

# Access detected boxes and class names
boxes = results.boxes  # Bounding boxes
names = results.names  # Class names mapping (e.g., {0: 'person', 1: 'bicycle', ...})

# Get the predicted class IDs for all detected objects
detected_class_ids = boxes.cls.tolist()

# Map class IDs to class names
detected_objects = [names[int(class_id)] for class_id in detected_class_ids]

# Count the occurrences of each detected object
object_counts = Counter(detected_objects)

print("Detected Objects and Quantities:")
for obj, count in object_counts.items():
    print(f"- {obj}: {count}")

# If you want to see individual detections with confidence:
print("\nIndividual Detections (Object - Confidence):")
for i, (box, class_id) in enumerate(zip(boxes.xyxy, detected_class_ids)):
    confidence = boxes.conf[i].item()
    object_name = names[int(class_id)]
    print(f"  {object_name} (Confidence: {confidence:.2f})")

In [ ]:
import cv2

custom_model_file = "./best.pt"
# custom_model_file = "yolov8n.pt"

input_image_file = "../images/horses.jpg"

print(f"Loading model from: {custom_model_file}")
model = YOLO(custom_model_file)

# 2. Run prediction on the image
# 'save=True' will save the annotated image to 'runs/detect/predictX/'
print(f"Running prediction on: {input_image_file}")
prediction_detection = model(
    source=input_image_file, save=True, show=False, tracker="bytetrack.yaml"
)

# prediction_detection = model(source = cv2.cvtColor(input_image_file, cv2.COLOR_BGR2RGB), save=True, show=False)

In [ ]:
from collections import Counter

# prediction_detection is a list of Results objects (one for each image in the batch)
# Since you're predicting on a single image, you'll access the first (and only) Results object
results = prediction_detection[0]

# Access detected boxes and class names
boxes = results.boxes  # Bounding boxes
names = results.names  # Class names mapping (e.g., {0: 'person', 1: 'bicycle', ...})

# Get the predicted class IDs for all detected objects
detected_class_ids = boxes.cls.tolist()

# Map class IDs to class names
detected_objects = [names[int(class_id)] for class_id in detected_class_ids]

# Count the occurrences of each detected object
object_counts = Counter(detected_objects)

print("Detected Objects and Quantities:")
for obj, count in object_counts.items():
    print(f"- {obj}: {count}")

# If you want to see individual detections with confidence:
print("\nIndividual Detections (Object - Confidence):")
for i, (box, class_id) in enumerate(zip(boxes.xyxy, detected_class_ids)):
    confidence = boxes.conf[i].item()
    object_name = names[int(class_id)]
    print(f"  {object_name} (Confidence: {confidence:.2f})")

In [ ]:
from collections import defaultdict
import cv2
import numpy as np
from ultralytics import YOLO

custom_model_file = "./best.pt"
model = YOLO(custom_model_file)

# Open the video file
video_path = "./fish_video.mp4"
cap = cv2.VideoCapture(video_path)

# Store the track history
track_history = defaultdict(lambda: [])

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO11 tracking on the frame, persisting tracks between frames
        result = model.track(frame, persist=True)[0]

        # Get the boxes and track IDs
        if result.boxes and result.boxes.is_track:
            boxes = result.boxes.xywh.cpu()
            track_ids = result.boxes.id.int().cpu().tolist()

            # Visualize the result on the frame
            frame = result.plot()

            # Plot the tracks
            for box, track_id in zip(boxes, track_ids):
                x, y, w, h = box
                track = track_history[track_id]
                track.append((float(x), float(y)))  # x, y center point
                if len(track) > 30:  # retain 30 tracks for 30 frames
                    track.pop(0)

                # Draw the tracking lines
                # points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                # cv2.polylines(
                #     frame, [points], isClosed=False, color=(230, 230, 230), thickness=10
                # )

        # Display the annotated frame
        cv2.imshow("YOLO11 Tracking", frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()

: 